# Comparación de modelos para la clasificación de residuos (3 clases)

**Objetivo**: comparar varias arquitecturas con **los mismos datos, la misma división y la misma receta de entrenamiento**
que el modelo actual (`CNN_Waste.ipynb`), para justificar con datos cuál es el mejor modelo para EcoSort.

| Modelo | Tipo | Por qué se compara |
|---|---|---|
| CNN desde cero | Red convolucional propia, sin preentrenar | Línea base: muestra cuánto aporta el transfer learning |
| MobileNetV3-Large | Transfer learning (ImageNet) | Modelo liviano pensado para celulares |
| ResNet-50 | Transfer learning (ImageNet) | El modelo clásico más usado en la literatura de residuos |
| EfficientNet-B0 | Transfer learning (ImageNet) | El modelo actual de EcoSort |

Métricas: exactitud, exactitud balanceada, macro-F1, recall por clase, número de parámetros,
tiempo de entrenamiento y tiempo de inferencia por imagen (GPU y CPU).

## 1. Configuración y librerías

In [ ]:
import os, random, glob, json, time, re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, confusion_matrix

SEED = 42
def fijar_semillas(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
fijar_semillas()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
torch.backends.cudnn.benchmark = True

RESULTS_DIR = '../resultados'
WEIGHTS_DIR = '../resultados/pesos_comparacion'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

## 2. Datos: misma división que el modelo actual
Se repite exactamente el código de `CNN_Waste.ipynb` (misma semilla, mismos límites de ropa y zapatos,
mismo split estratificado por clase y fuente), así todos los modelos se evalúan con las **mismas 5.997 imágenes de prueba**.

In [ ]:
DATA_ROOT = '../datasets/waste_data'
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
NUM_WORKERS = 4
SOURCE_CAPS = {'clothes': 700, 'shoes': 500}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def source_of(path):
    # Extrae el prefijo del nombre del archivo (todo antes del primer número)
    name = os.path.basename(path).lower()
    m = re.match(r'^([a-z\-]+?)[_\s]?\d', name)
    return m.group(1) if m else 'otro'

train_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomAffine(degrees=20, translate=(0.1, 0.1), shear=10, scale=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
eval_full = datasets.ImageFolder(DATA_ROOT, transform=eval_tf)
targets = np.array(train_full.targets)
sources = np.array([source_of(p) for p, _ in train_full.samples])

# Límites por fuente (misma muestra aleatoria que el notebook original)
rng = np.random.default_rng(SEED)
keep = np.ones(len(targets), dtype=bool)
for src, cap in SOURCE_CAPS.items():
    idx = np.where(sources == src)[0]
    if len(idx) > cap:
        keep[rng.choice(idx, len(idx) - cap, replace=False)] = False
indices = np.where(keep)[0]

# Split estratificado por clase + fuente (70 / 15 / 15)
strata = np.array([f'{t}|{s}' for t, s in zip(targets[indices], sources[indices])])
train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=strata, random_state=SEED)
strata_temp = np.array([f'{targets[i]}|{sources[i]}' for i in temp_idx])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=strata_temp, random_state=SEED)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(Subset(train_full, train_idx), shuffle=True, **loader_kwargs)
val_loader = DataLoader(Subset(eval_full, val_idx), shuffle=False, **loader_kwargs)
test_loader = DataLoader(Subset(eval_full, test_idx), shuffle=False, **loader_kwargs)

CLASS_NAMES = train_full.classes
NUM_CLASSES = len(CLASS_NAMES)
LABELS_EN = {'Norecyclable': 'Non-recyclable', 'organic': 'Organic', 'recyclable': 'Recyclable'}
print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')
print('Clases:', train_full.class_to_idx)

# Pesos por clase para compensar el desbalance (igual que el modelo actual)
train_counts = np.bincount(targets[train_idx], minlength=NUM_CLASSES)
class_weights = torch.tensor(train_counts.sum() / (NUM_CLASSES * train_counts), dtype=torch.float32, device=device)
print('Pesos por clase:', dict(zip(CLASS_NAMES, class_weights.round(decimals=3).tolist())))

## 3. Modelos
Cada función devuelve el modelo, la **cabeza** (capas nuevas que se entrenan en la fase 1)
y los **últimos bloques** que se descongelan en la fase 2 (fine-tuning).
La CNN desde cero no tiene pesos preentrenados, así que se entrena completa en una sola fase.

In [ ]:
class CNNBaseline(nn.Module):
    # Misma CNN desde cero del notebook de animales: 4 bloques convolucionales + Global Average Pooling
    def __init__(self, num_classes):
        super().__init__()
        def bloque(c_in, c_out):
            return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(bloque(3, 32), bloque(32, 64), bloque(64, 128), bloque(128, 256))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
                                        nn.Dropout(0.5), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(torch.flatten(self.gap(self.features(x)), 1))

def build_model(nombre):
    if nombre == 'CNN desde cero':
        m = CNNBaseline(NUM_CLASSES)
        return m, None, None
    if nombre == 'MobileNetV3-Large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
        return m, m.classifier, list(m.features[-4:])
    if nombre == 'ResNet-50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, NUM_CLASSES))
        return m, m.fc, [m.layer4]
    if nombre == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
        return m, m.classifier, list(m.features[-3:])
    raise ValueError(nombre)

MODELOS = ['CNN desde cero', 'MobileNetV3-Large', 'ResNet-50', 'EfficientNet-B0']
for n in MODELOS:
    m, _, _ = build_model(n)
    print(f'{n:18s} {sum(p.numel() for p in m.parameters()) / 1e6:6.2f} M parámetros')

## 4. Entrenamiento (misma receta para todos)
- **Transfer learning**: fase 1 = solo la cabeza, 5 épocas, Adam lr=1e-3; fase 2 = últimos bloques, hasta 12 épocas, lr=1e-4.
- **CNN desde cero**: todas las capas, hasta 25 épocas, Adam lr=1e-3.
- En todos: pesos por clase, mixed precision, ReduceLROnPlateau, early stopping (paciencia 4) y se guarda el mejor modelo según la pérdida de validación.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler('cuda')

def run_epoch(model, loader, optimizer=None):
    # Una época de entrenamiento (si hay optimizador) o de evaluación
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def train_phase(model, params, lr, epochs, nombre, fase, state):
    optimizer = optim.Adam(params, lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    paciencia = 0
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        scheduler.step(va_loss)
        state['history'].append({'fase': fase, 'train_loss': tr_loss, 'val_loss': va_loss, 'train_acc': tr_acc, 'val_acc': va_acc})
        if va_loss < state['best_loss']:
            state['best_loss'] = va_loss
            state['best_state'] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            paciencia = 0
        else:
            paciencia += 1
        print(f'{nombre} | {fase} {ep+1}/{epochs} | train {tr_loss:.4f}/{tr_acc:.4f} | val {va_loss:.4f}/{va_acc:.4f} | {time.time()-t0:.0f}s', flush=True)
        if paciencia >= 4:
            print('Early stopping.', flush=True)
            break

def entrenar(nombre):
    fijar_semillas()
    model, head, last_blocks = build_model(nombre)
    model = model.to(device)
    state = {'best_loss': float('inf'), 'best_state': None, 'history': []}
    t0 = time.time()
    if head is None:
        # CNN desde cero: se entrena completa
        train_phase(model, model.parameters(), 1e-3, 25, nombre, 'completo', state)
    else:
        # Fase 1: base congelada, solo la cabeza
        for p in model.parameters(): p.requires_grad = False
        for p in head.parameters(): p.requires_grad = True
        train_phase(model, [p for p in model.parameters() if p.requires_grad], 1e-3, 5, nombre, 'cabeza', state)
        # Fase 2: fine-tuning de los últimos bloques + cabeza
        for block in last_blocks:
            for p in block.parameters(): p.requires_grad = True
        train_phase(model, [p for p in model.parameters() if p.requires_grad], 1e-4, 12, nombre, 'fine-tuning', state)
    train_time = time.time() - t0
    model.load_state_dict(state['best_state'])
    torch.save(state['best_state'], os.path.join(WEIGHTS_DIR, nombre.replace(' ', '_') + '.pth'))
    return model, train_time, state['history']

## 5. Evaluación en el conjunto de prueba

In [ ]:
def evaluar(model):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            with torch.autocast(device_type='cuda'):
                out = model(images.to(device, non_blocking=True))
            probs.append(F.softmax(out.float(), 1).cpu().numpy()); ys.append(labels.numpy())
    probs, ys = np.concatenate(probs), np.concatenate(ys)
    pred = probs.argmax(1)
    test_sources = sources[test_idx]
    def acierto(objeto):
        mask = test_sources == objeto
        return float((pred[mask] == ys[mask]).mean()) if mask.any() else None
    masks = (test_sources == 'trash') | (test_sources == 'trashbox-masks')
    return {
        'accuracy': float((pred == ys).mean()),
        'balanced_accuracy': float(balanced_accuracy_score(ys, pred)),
        'macro_f1': float(f1_score(ys, pred, average='macro')),
        'recall_por_clase': {LABELS_EN[c]: float(r) for c, r in zip(CLASS_NAMES, recall_score(ys, pred, average=None))},
        'objetos': {'Batteries': acierto('battery'), 'Cell phones': acierto('trashbox-smartphones'),
                    'Face masks': float((pred[masks] == ys[masks]).mean()), 'Cigarette butts': acierto('trashbox-cigarette-butt'),
                    'Cardboard (TrashBox)': acierto('trashbox-cardboard')},
        'confusion_matrix': confusion_matrix(ys, pred).tolist(),
    }

def tiempo_inferencia(model, dev, repeticiones):
    # Tiempo promedio para clasificar UNA imagen (lo que hace la página web en cada petición)
    model = model.to(dev).eval()
    x = torch.randn(1, 3, *IMG_SIZE, device=dev)
    with torch.no_grad():
        for _ in range(10): model(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(repeticiones): model(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeticiones * 1000  # milisegundos

## 6. Entrenar y evaluar los cuatro modelos

In [ ]:
resultados, historiales = {}, {}
for nombre in MODELOS:
    print(f'\n========== {nombre} ==========', flush=True)
    model, train_time, history = entrenar(nombre)
    r = evaluar(model)
    r['parametros_M'] = sum(p.numel() for p in model.parameters()) / 1e6
    r['tiempo_entrenamiento_min'] = train_time / 60
    r['epocas'] = len(history)
    r['inferencia_gpu_ms'] = tiempo_inferencia(model, torch.device('cuda'), 100)
    r['inferencia_cpu_ms'] = tiempo_inferencia(model, torch.device('cpu'), 20)
    resultados[nombre] = r
    historiales[nombre] = history
    print(f"RESULTADO {nombre}: acc {r['accuracy']:.4f} | bal {r['balanced_accuracy']:.4f} | F1 {r['macro_f1']:.4f} | "
          f"{r['parametros_M']:.1f}M | {r['tiempo_entrenamiento_min']:.1f} min | GPU {r['inferencia_gpu_ms']:.1f} ms | CPU {r['inferencia_cpu_ms']:.1f} ms", flush=True)
    # Se guarda después de cada modelo para no perder nada si algo falla
    with open(os.path.join(RESULTS_DIR, 'comparacion_modelos.json'), 'w', encoding='utf-8') as f:
        json.dump({'resultados': resultados, 'historiales': historiales}, f, ensure_ascii=False, indent=2)
    del model; torch.cuda.empty_cache()

## 7. Tabla comparativa

In [ ]:
tabla = pd.DataFrame({
    n: {'Accuracy (%)': r['accuracy'] * 100, 'Balanced acc. (%)': r['balanced_accuracy'] * 100, 'Macro-F1 (%)': r['macro_f1'] * 100,
        'Recall non-recyclable (%)': r['recall_por_clase']['Non-recyclable'] * 100, 'Batteries (%)': r['objetos']['Batteries'] * 100,
        'Params (M)': r['parametros_M'], 'Training (min)': r['tiempo_entrenamiento_min'],
        'GPU (ms/img)': r['inferencia_gpu_ms'], 'CPU (ms/img)': r['inferencia_cpu_ms']}
    for n, r in resultados.items()}).T.round(2)
tabla.to_csv(os.path.join(RESULTS_DIR, 'comparacion_modelos.csv'))
display(tabla)

## 8. Gráficas

In [ ]:
nombres = list(resultados)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(nombres)); w = 0.27
for i, (k, lab) in enumerate([('accuracy', 'Accuracy'), ('balanced_accuracy', 'Balanced accuracy'), ('macro_f1', 'Macro-F1')]):
    axes[0].bar(x + (i - 1) * w, [resultados[n][k] * 100 for n in nombres], w, label=lab)
axes[0].set_xticks(x); axes[0].set_xticklabels(nombres); axes[0].set_ylabel('%'); axes[0].legend()
axes[0].set_ylim(min(resultados[n]['accuracy'] for n in nombres) * 100 - 10, 100)
axes[0].set_title('Desempeño en el conjunto de prueba')

for n in nombres:
    axes[1].scatter(resultados[n]['inferencia_cpu_ms'], resultados[n]['macro_f1'] * 100, s=resultados[n]['parametros_M'] * 20 + 40)
    axes[1].annotate(n, (resultados[n]['inferencia_cpu_ms'], resultados[n]['macro_f1'] * 100), textcoords='offset points', xytext=(8, 4))
axes[1].set_xlabel('Tiempo por imagen en CPU (ms)'); axes[1].set_ylabel('Macro-F1 (%)')
axes[1].set_title('Calidad vs. velocidad (tamaño = parámetros)')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(nombres), figsize=(4.2 * len(nombres), 3.2), sharey=True)
for ax, n in zip(axes, nombres):
    ax.plot([h['val_acc'] for h in historiales[n]], label='Validación')
    ax.plot([h['train_acc'] for h in historiales[n]], label='Entrenamiento')
    ax.set_title(n); ax.set_xlabel('Época')
axes[0].set_ylabel('Accuracy'); axes[0].legend()
plt.tight_layout(); plt.show()